# 🔍 Live Anomaly Detection Demo

This notebook demonstrates the full pipeline:  
**Raw vibration signal → Preprocessing → TFLite Model → Anomaly Decision**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
import scipy.io as sio
import os
import time
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)

TFLITE_MODEL = '../models/cae_quantized.tflite'
DATA_NPZ     = '../data/processed/cwru_processed.npz'
CWRU_DIR     = '../CWRU/raw'
SCALER_MEAN  = '../data/processed/cwru_scaler_mean.npy'
SCALER_SCALE = '../data/processed/cwru_scaler_scale.npy'

WINDOW_SIZE = 2048
SR          = 48000

# load scaler and model
scaler_mean  = np.load(SCALER_MEAN)
scaler_scale = np.load(SCALER_SCALE)

interp = tf.lite.Interpreter(model_path=TFLITE_MODEL)
interp.allocate_tensors()
IN_IDX  = interp.get_input_details()[0]['index']
OUT_IDX = interp.get_output_details()[0]['index']

# load threshold from training (p95 of normal MSE)
data       = np.load(DATA_NPZ, allow_pickle=True)
X_all      = data['X_raw'][..., np.newaxis].astype(np.float32)
y          = data['y_labels']

mse_n = []
for i in np.where(y == 0)[0]:
    s = X_all[i:i+1]
    interp.set_tensor(IN_IDX, s)
    interp.invoke()
    r = interp.get_tensor(OUT_IDX)
    mse_n.append(float(np.mean((s-r)**2)))

THRESHOLD = np.percentile(mse_n, 95)
print(f'✅  Model loaded  —  Threshold = {THRESHOLD:.6f}')

---
## Demo 1 — Test on a SINGLE WINDOW (you choose)

In [ ]:
# ── CHANGE THESE TWO LINES TO TEST DIFFERENT INPUTS ──────────────
# Options: 'Normal', 'Ball_007', 'Ball_014', 'Ball_021',
#          'InnerRace_007', 'InnerRace_014', 'InnerRace_021'
#          'OuterRace_007', 'OuterRace_014', 'OuterRace_021'
TEST_CLASS  = 'Normal'    # change this
WINDOW_NUM  = 0           # which window to pick from that class
# ─────────────────────────────────────────────────────────────────

y_names = data['y_names']
mask    = (y_names == TEST_CLASS)
indices = np.where(mask)[0]

if WINDOW_NUM >= len(indices):
    print(f'Only {len(indices)} windows for {TEST_CLASS}. Using 0.')
    WINDOW_NUM = 0

idx     = indices[WINDOW_NUM]
sample  = X_all[idx:idx+1]   # shape (1, 2048, 1)

# run inference
t0 = time.perf_counter()
interp.set_tensor(IN_IDX, sample)
interp.invoke()
recon  = interp.get_tensor(OUT_IDX)
lat_ms = (time.perf_counter() - t0) * 1000

score    = float(np.mean((sample - recon)**2))
is_fault = score > THRESHOLD

# ─── result display ───
color  = '#F44336' if is_fault else '#4CAF50'
status = '🔴 ANOMALY DETECTED' if is_fault else '🟢 NORMAL'

print(f'Input class   : {TEST_CLASS}')
print(f'Anomaly score : {score:.6f}')
print(f'Threshold     : {THRESHOLD:.6f}')
print(f'Latency       : {lat_ms:.2f} ms')
print(f'Decision      : {status}')

# plot original vs reconstructed
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(sample[0, :, 0], linewidth=0.6, color='steelblue', label='Input signal')
axes[0].plot(recon[0,  :, 0], linewidth=0.6, color=color, linestyle='--',
              label=f'Reconstruction  (MSE={score:.6f})')
axes[0].set_title(f'Class: {TEST_CLASS}  |  Decision: {status}', fontsize=11)
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Amplitude')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# error signal
err = np.abs(sample[0, :, 0] - recon[0, :, 0])
axes[1].fill_between(range(len(err)), err, color=color, alpha=0.5, label='|Error|')
axes[1].axhline(np.sqrt(THRESHOLD), color='orange', linestyle='--', label='Threshold level')
axes[1].set_title('Reconstruction Error Signal')
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('|Error|')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Demo 2 — Test all fault types at once (full comparison)

In [ ]:
ALL_CLASSES = ['Normal','Ball_007','Ball_014','Ball_021',
               'InnerRace_007','InnerRace_014','InnerRace_021',
               'OuterRace_007','OuterRace_014','OuterRace_021']

results = []
for cls in ALL_CLASSES:
    mask = (y_names == cls)
    if not mask.any():
        continue
    idx    = np.where(mask)[0][0]
    samp   = X_all[idx:idx+1]

    interp.set_tensor(IN_IDX, samp)
    interp.invoke()
    rec    = interp.get_tensor(OUT_IDX)
    score  = float(np.mean((samp - rec)**2))
    detected = score > THRESHOLD

    results.append({'class': cls, 'score': score, 'detected': detected,
                    'expected_fault': cls != 'Normal'})

print(f'\n{"Class":<20} {"Score":>10} {"Detected?":>12} {"Correct?":>10}')
print('-' * 56)
correct = 0
for r in results:
    status = '🔴 FAULT' if r['detected'] else '🟢 Normal'
    ok     = '✅' if r['detected'] == r['expected_fault'] else '❌'
    if r['detected'] == r['expected_fault']:
        correct += 1
    print(f"{r['class']:<20} {r['score']:>10.6f} {status:>12}  {ok}")

print(f'\nAccuracy on these samples: {correct}/{len(results)} = {correct/len(results)*100:.0f}%')

In [ ]:
# visual bar chart of scores vs threshold
classes = [r['class'] for r in results]
scores  = [r['score'] for r in results]
colors  = ['#4CAF50' if not r['expected_fault'] else '#F44336' for r in results]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(classes, scores, color=colors, alpha=0.8)
ax.axhline(THRESHOLD, color='orange', linewidth=2.5, linestyle='--',
            label=f'Threshold = {THRESHOLD:.5f}')

for bar, s, r in zip(bars, scores, results):
    marker = '✅' if r['detected'] == r['expected_fault'] else '❌'
    ax.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + THRESHOLD*0.02,
             f'{s:.4f}\n{marker}', ha='center', fontsize=7.5)

ax.set_xticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Reconstruction MSE (Anomaly Score)')
ax.set_title('Anomaly Score vs Threshold — All Fault Types')

green_patch = mpatches.Patch(color='#4CAF50', alpha=0.8, label='Normal')
red_patch   = mpatches.Patch(color='#F44336', alpha=0.8, label='Fault')
ax.legend(handles=[green_patch, red_patch,
                    plt.Line2D([0],[0], color='orange', linestyle='--', linewidth=2,
                                label=f'Threshold={THRESHOLD:.5f}')], fontsize=8)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/demo_all_faults.png', dpi=150)
plt.show()

---
## Demo 3 — Load a RAW .mat file and test directly

In [ ]:
# ── CHANGE THIS TO ANY .mat FILE ─────────────────────────────────
TEST_FILE = '../CWRU/raw/IR021_1_214.mat'   # change filename here
# ─────────────────────────────────────────────────────────────────

def load_mat_signal(filepath):
    mat = sio.loadmat(filepath)
    for k in mat.keys():
        if 'DE_time' in k:
            return mat[k].flatten().astype(np.float32)
    for k in mat.keys():
        if 'time' in k.lower() and not k.startswith('__'):
            return mat[k].flatten().astype(np.float32)

def run_on_raw_file(filepath):
    signal = load_mat_signal(filepath)

    # take middle window
    mid    = len(signal)//2
    window = signal[mid:mid + WINDOW_SIZE]

    # normalise
    norm   = (window - scaler_mean) / scaler_scale
    x      = norm[np.newaxis, :, np.newaxis].astype(np.float32)

    # inference
    t0 = time.perf_counter()
    interp.set_tensor(IN_IDX, x)
    interp.invoke()
    r  = interp.get_tensor(OUT_IDX)
    ms = (time.perf_counter() - t0) * 1000

    score    = float(np.mean((x - r)**2))
    is_fault = score > THRESHOLD

    # plot
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    color = '#F44336' if is_fault else '#4CAF50'
    status = '🔴 ANOMALY' if is_fault else '🟢 NORMAL'

    axes[0].plot(window, linewidth=0.6, color='steelblue', label='Raw signal')
    axes[0].set_title(f'File: {os.path.basename(filepath)}  |  Decision: {status}  |  MSE: {score:.6f}', fontsize=10)
    axes[0].set_xlabel('Sample'); axes[0].set_ylabel('Amplitude')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(x[0,:,0], linewidth=0.6, color='steelblue', label='Normalised input')
    axes[1].plot(r[0,:,0], linewidth=0.6, color=color, linestyle='--', label='Reconstruction')
    axes[1].set_xlabel('Sample'); axes[1].set_ylabel('Normalised amplitude')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f'File    : {os.path.basename(filepath)}')
    print(f'Score   : {score:.6f}  (threshold={THRESHOLD:.6f})')
    print(f'Latency : {ms:.2f} ms')
    print(f'Result  : {status}')

run_on_raw_file(TEST_FILE)

---
## Demo 4 — MIMII Audio Anomaly Detection

In [ ]:
import librosa

MIMII_NPZ    = '../data/processed/mimii_processed.npz'
KERAS_MODEL  = '../models/cae_model.h5'

mimii  = np.load(MIMII_NPZ, allow_pickle=True)
X_norm = mimii['X_val']
X_ab   = mimii['X_abnormal']

# use keras model for 2D spectrograms
# (separate 2D CAE needed — using reconstruction error as proxy)
# For demo: show spectrogram + MSE of 2 samples

MIMII_NORM_DIR   = '../MIMII/normal'
MIMII_ABNORM_DIR = '../MIMII/abnormal'

def mel_from_file(path):
    audio, _ = librosa.load(path, sr=16000, mono=True)
    audio    = audio[:160000]   # 10 seconds
    mel = librosa.feature.melspectrogram(y=audio, sr=16000,
                                          n_fft=1024, hop_length=512, n_mels=64)
    return librosa.power_to_db(mel, ref=np.max)

norm_files  = sorted(os.listdir(MIMII_NORM_DIR))[:3]
abnorm_files = sorted(os.listdir(MIMII_ABNORM_DIR))[:3]

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for i, (fname, ax) in enumerate(zip(norm_files, axes[0])):
    mel = mel_from_file(os.path.join(MIMII_NORM_DIR, fname))
    librosa.display.specshow(mel, sr=16000, hop_length=512, ax=ax)
    ax.set_title(f'NORMAL — {fname}', fontsize=8, color='green')

for i, (fname, ax) in enumerate(zip(abnorm_files, axes[1])):
    mel = mel_from_file(os.path.join(MIMII_ABNORM_DIR, fname))
    librosa.display.specshow(mel, sr=16000, hop_length=512, ax=ax)
    ax.set_title(f'ABNORMAL — {fname}', fontsize=8, color='red')

plt.suptitle('MIMII Machine Sound — Log-Mel Spectrograms', fontsize=11)
plt.tight_layout()
plt.savefig('../results/plots/demo_mimii.png', dpi=150)
plt.show()

---
## Demo 5 — Sliding Window Real-time Simulation

In [ ]:
# Simulate what happens when a continuous vibration signal comes in
# and we slide the detection window over it

# ── CHANGE THIS TO 'Normal' or 'InnerRace_021' etc ───────────────
SIM_CLASS = 'OuterRace_021'
# ─────────────────────────────────────────────────────────────────

FILE_MAP = {
    'Normal'        : 'Time_Normal_1_098.mat',
    'Ball_007'      : 'B007_1_123.mat',
    'Ball_021'      : 'B021_1_227.mat',
    'InnerRace_007' : 'IR007_1_110.mat',
    'InnerRace_021' : 'IR021_1_214.mat',
    'OuterRace_007' : 'OR007_6_1_136.mat',
    'OuterRace_021' : 'OR021_6_1_239.mat',
}

raw_signal = load_mat_signal(os.path.join(CWRU_DIR, FILE_MAP[SIM_CLASS]))

STEP       = 1024    # slide by half window each time
N_WINDOWS  = 30      # simulate 30 consecutive windows

scores = []
flags  = []
for i in range(N_WINDOWS):
    start  = i * STEP
    window = raw_signal[start:start + WINDOW_SIZE]
    norm   = (window - scaler_mean) / scaler_scale
    x      = norm[np.newaxis, :, np.newaxis].astype(np.float32)
    interp.set_tensor(IN_IDX, x)
    interp.invoke()
    r = interp.get_tensor(OUT_IDX)
    s = float(np.mean((x - r)**2))
    scores.append(s)
    flags.append(s > THRESHOLD)

# plot
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

n_pts = N_WINDOWS * STEP + WINDOW_SIZE
t_ms  = np.arange(min(n_pts, len(raw_signal))) / SR * 1000
axes[0].plot(t_ms, raw_signal[:len(t_ms)], linewidth=0.5, color='steelblue')
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'Input Signal — {SIM_CLASS}')
axes[0].grid(True, alpha=0.3)

win_times = [(i * STEP + WINDOW_SIZE//2)/SR*1000 for i in range(N_WINDOWS)]
colors_s  = ['red' if f else 'green' for f in flags]
axes[1].bar(win_times, scores, width=STEP/SR*1000*0.8, color=colors_s, alpha=0.75)
axes[1].axhline(THRESHOLD, color='orange', linestyle='--', linewidth=2,
                 label=f'Threshold={THRESHOLD:.5f}')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Anomaly Score (MSE)')
axes[1].set_title('Sliding Window Detection — Green=Normal, Red=Fault')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/sliding_window_demo.png', dpi=150)
plt.show()

n_fault = sum(flags)
print(f'Class: {SIM_CLASS}')
print(f'Windows flagged as FAULT: {n_fault}/{N_WINDOWS} ({n_fault/N_WINDOWS*100:.0f}%)')